# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for exploring the FAIR² dataset using the `mlcroissant` library. The dataset comprises a rich set of tabular records with clinical, molecular, and pathological variables for cancer survivors with second primary colorectal cancer (CRC).

### Dataset Source
The Croissant schema describing this dataset is available at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Show core metadata as an overview
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields in each record set. All exploration below strictly uses `@id` values.

In [ ]:
# Discover available record sets and their fields
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
record_set_ids = []
for rs in record_sets:
    print(f"• Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields in this set:")
    for field in rs.fields:
        print(f"    - {field.name}, @id: {field.id}, type: {field.data_type}")
    print()
    record_set_ids.append(rs.id)

# Preview first 2 records from each record set
for rs in record_sets:
    print(f"First 2 records from record set '{rs.name}' (@id: {rs.id}):")
    for idx, record in enumerate(dataset.records(record_set=rs.id)):
        print(f"  Record {idx+1}: {record}")
        if idx >= 1:
            break
    print()

## 3. Data Extraction
Load all data from the primary tabular record set(s) as DataFrames for analysis. Only the record set and field `@id` values, found above, are used for reference.

In [ ]:
# For this dataset, typically there is only 1 record set. If more, adjust as needed.

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set @id: {rs_id}, shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Demonstrate common processing steps, such as filtering, normalizing, and grouping using data field `@id`s. Let's analyze a numeric field and a categorical field found in section 2 above:

In [ ]:
# Select primary record set and IDs for numeric/categorical fields.
# Adjust the field IDs below as needed, per your overview in section 2.

# Example (adjust as appropriate if a different @id was printed above):
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Print available columns with their @id
print("Columns present in the main record set:")
for col in df.columns:
    print(f" - {col}")

# Pick a numeric field (such as 'cr:field_Age', or whichever @id for age is present)
numeric_field_id = None
for col in df.columns:
    if 'Age' in col or col.lower() == 'cr:field_age':
        numeric_field_id = col
        break
if not numeric_field_id:
    # If Age is not available, just use the first float/int field (guessing by name)
    for col in df.columns:
        if 'age' in col.lower() or 'interval' in col.lower() or 'number' in col.lower():
            numeric_field_id = col
            break
assert numeric_field_id, "No numeric field found. Please adjust this selection."

# Pick a grouping/categorical field (for example, Sex or MSI_H_status)
group_field_id = None
possible_group_fields = ['cr:field_Sex', 'cr:field_sex', 'cr:field_MSI_status', 'cr:field_MSI-H', 'cr:field_anatomical_location']
for candidate in possible_group_fields:
    if candidate in df.columns:
        group_field_id = candidate
        break
if not group_field_id:
    # Just pick any field that is object type and non-numeric
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break
assert group_field_id, "No suitable grouping field found. Please adjust this selection."

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Convert numeric column to numeric dtype
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove outliers by filtering on plausible age (if 'age'), else use quantile filter
if 'age' in numeric_field_id.lower():
    threshold = 30  # only those over 30 years
    filtered_df = df[df[numeric_field_id] > threshold]
else:
    threshold = df[numeric_field_id].quantile(0.2)
    filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records where {numeric_field_id} > {threshold}:")
display(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id}, preview:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by the group_field and compute means
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df)
else:
    print(f"Grouping field {group_field_id} not in DataFrame.")

## 5. Visualization
Visualize the distribution of the numeric field, and its relationship to your group field, using standard plotting libraries.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group
plt.figure(figsize=(8,4))
sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
plt.title(f"{numeric_field_id} by {group_field_id}")
plt.show()

## 6. Conclusion

In this notebook, you used the `mlcroissant` library to explore the FAIR² colorectal cancer dataset, referencing all entities by their canonical `@id`. After loading and reviewing the dataset, you extracted the data into DataFrames, performed EDA (including filtering and normalization), and generated key visualizations to uncover relationships between core variables. This workflow provides a reproducible and standards-driven approach for exploring any Croissant-encoded dataset.